# Load Packages

In [1]:
import os
import pandas as pd
from transformers import pipeline

In [2]:
# Custom functions
from custom_functions import (
    get_device,
    get_pipeline_device_id,
    analyze_sentiment,
)

# Parameters

In [3]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Load Processed Data

In [4]:
# Load dataset
toaster_dedup_df = pd.read_csv(os.path.join(data_processed_dir, "toaster_dedup.csv"))

# Data preview
toaster_dedup_df.head()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,RV_DT,VP,HLP_VT,IMG_PRST,TTL_RV,RVS_L,RV_TRANS,SUBJ,SRVS,CP_RVS
0,B01KZ729F6,Hamilton Beach 2 Slice Extra Wide Slot Toaster...,NaN,0.220000,24\n.\n99,1,0,4.4,12579,https://www.amazon.com/stores/HamiltonBeach/pa...,...,2019-01-06,1,.,0,1669,31,Love making 4 slices at a time.,0.6,positive,0.6369
1,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Eas...,209.988477,0.786655,44.8,1,0,4.1,4156,https://www.amazon.com/stores/Nostalgia/page/B...,...,2018-12-09,1,.,0,863,96,Great item to use if you love grilled cheese b...,0.675,positive,0.8519
2,B0BT5WXBR2,Elite Gourmet ECT118B Cool Touch Single Slice ...,14.990000,0.000000,14.99,1,0,.,.,https://www.amazon.com/stores/EliteGourmet/pag...,...,2021-03-28,1,.,0,4078,79,I had high hopes for this toaster - but it ta...,0.32,positive,0.4404
3,B0B9MX21NV,"evoloop Toaster 2 Slice, Stainless Steel Bread...",279.850020,0.874969,34.99,1,0,4.4,31,https://www.amazon.com/stores/evoloop/page/087...,...,2022-11-17,0,1,1,61,3565,BE AWARE: Read all the instructions included a...,0.509831254980509,positive,0.9914
4,B00ZGCKSG8,DASH Clear View Toaster: Extra Wide Slot Toast...,NaN,0.170000,41\n.\n48,1,0,4.4,11283,https://www.amazon.com/stores/DASH/page/F42BA3...,...,2020-10-04,1,35,0,3224,.,.,.,.,.


In [5]:
toaster_dedup_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62014 entries, 0 to 62013
Data columns (total 30 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ASIN         62014 non-null  str    
 1   P_TITLE      62014 non-null  str    
 2   OP           45643 non-null  float64
 3   DP           62014 non-null  float64
 4   SP           62014 non-null  str    
 5   FS           62014 non-null  str    
 6   PRA_4.5      62014 non-null  int64  
 7   P_RTG        62014 non-null  str    
 8   RTG_P_NO     62014 non-null  str    
 9   SELLER_LINK  62014 non-null  str    
 10  IMAGE_URL    62014 non-null  str    
 11  P_URL        62014 non-null  str    
 12  RV_URL       62014 non-null  str    
 13  PRFL_IMG     62014 non-null  str    
 14  PRFL_URL     62014 non-null  str    
 15  RV_TTL       62011 non-null  str    
 16  RVS          62008 non-null  str    
 17  RVR          62011 non-null  str    
 18  RSR          62014 non-null  int64  
 19  RVR_CONT     62

# Sentiment Analysis

In [6]:
# take a sample to speed up testing — remove or increase for full analysis
# toaster_dedup_df = toaster_dedup_df.sample(50, random_state=42).copy()

## Device Detection
This will help detect if there exist a GPU on the device.

In [7]:
device = get_device()
device_id = get_pipeline_device_id(device)

Using Apple MPS (Apple M2 Max)


## Load Sentiment Pipeline

In [8]:
# Label Mapping
LABEL_MAP = {
    # Standard text labels
    "positive": "positive",
    "negative": "negative",
    "neutral":  "neutral",
    # Numeric labels (verify order on the model's HuggingFace card!)
    "label_0":  "negative",
    "label_1":  "neutral",
    "label_2":  "positive",
}

In [ ]:
# Different models to experiment with — each has its own strengths and weaknesses

# model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
# model_name = "distilbert-base-uncased-finetuned-sst-2-english"
# model_name = "cardiffnlp/twitter-roberta-base-sentiment"
# model_name = "microsoft/deberta-v3-base"
# model_name = "siebert/sentiment-roberta-large-english"
# model_name = "AnkitAI/reviews-roberta-base-sentiment-analysis"

### Twitter RoBERTa Base (CardiffNLP model)

In [10]:

model_twrb = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_pipeline_twrb = pipeline(
    "sentiment-analysis",
    model=model_twrb,
    tokenizer=model_twrb,
    device=device_id,
    truncation=True,        # Handles long reviews (truncates to 512 tokens)
    max_length=512,
    batch_size=32,          # Process in batches for speed
)
# Analyze sentiment and add results to the DataFrame
toaster_dedup_df = analyze_sentiment(
    toaster_dedup_df,
    sentiment_pipeline=sentiment_pipeline_twrb,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

# Rename BERT output columns for clarity
toaster_dedup_df = toaster_dedup_df.rename(
    columns={
        "SENTIMENT": "TWRB_SENT",
        "SENTIMENT_SCORE": "TWRB_SCORE",
    }
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Analyzing 61395 texts (skipping 619 empty/invalid)...

Short texts (direct batch): 61305
Long texts (chunking + voting): 90



Long texts: 100%|██████████| 90/90 [00:04<00:00, 18.72it/s]


Done. 90 text(s) used chunking + voting.


In [11]:
# Check distribution of sentiment labels and review the long reviews that were chunked
print("\nSentiment Distribution:")
display(toaster_dedup_df["TWRB_SENT"].value_counts())

print("\nLong reviews that used chunking:")
display(toaster_dedup_df[toaster_dedup_df["CHUNKED"]][["RV_TRANS", "TWRB_SENT", "TWRB_SCORE"]])


Sentiment Distribution:


TWRB_SENT
positive    37406
negative    17124
neutral      6865
Name: count, dtype: int64


Long reviews that used chunking:


,RV_TRANS,TWRB_SENT,TWRB_SCORE
3,BE AWARE: Read all the instructions included a...,neutral,0.6821
196,[UPDATED at bottom of review]\n---------------...,neutral,0.8554
741,"I just looked at the date we bought this, it w...",negative,0.7893
938,"Over the years, before today, we had purchased...",neutral,0.6841
1159,"Like so many others, I've recently started bak...",positive,0.8307
...,...,...,...
56008,"I am a fussy toast person, I like it just so.\...",positive,0.5268
58835,I own the Black & Decker TO1303SBD toaster ove...,positive,0.697
59673,I'd give it 5 stars but for a toaster this exp...,negative,0.6492
60085,"I wouldn't say that I'm a toast connoisseur, b...",positive,0.8089


### RoBERTa Large English (Siebert model)

In [12]:
model_srb = "siebert/sentiment-roberta-large-english"

sentiment_pipeline_srb = pipeline(
    "sentiment-analysis",
    model=model_srb,
    tokenizer=model_srb,
    device=device_id,
    truncation=True,        # Handles long reviews (truncates to 512 tokens)
    max_length=512,
    batch_size=32,          # Process in batches for speed
)
# Analyze sentiment and add results to the DataFrame
toaster_dedup_df = analyze_sentiment(
    toaster_dedup_df,
    sentiment_pipeline=sentiment_pipeline_srb,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

# Rename BERT output columns for clarity
toaster_dedup_df = toaster_dedup_df.rename(
    columns={
        "SENTIMENT": "SRB_SENT",
        "SENTIMENT_SCORE": "SRB_SCORE",
    }
)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (845 > 512). Running this sequence through the model will result in indexing errors


Analyzing 61395 texts (skipping 619 empty/invalid)...

Short texts (direct batch): 61305
Long texts (chunking + voting): 90



Long texts: 100%|██████████| 90/90 [00:12<00:00,  7.34it/s]


Done. 90 text(s) used chunking + voting.


In [13]:
# Check distribution of sentiment labels and review the long reviews that were chunked
print("\nSentiment Distribution:")
display(toaster_dedup_df["SRB_SENT"].value_counts())

print("\nLong reviews that used chunking:")
display(toaster_dedup_df[toaster_dedup_df["CHUNKED"]][["RV_TRANS", "SRB_SENT", "SRB_SCORE"]])


Sentiment Distribution:


SRB_SENT
positive    40456
negative    20939
Name: count, dtype: int64


Long reviews that used chunking:


,RV_TRANS,SRB_SENT,SRB_SCORE
3,BE AWARE: Read all the instructions included a...,positive,0.9978
196,[UPDATED at bottom of review]\n---------------...,negative,0.9988
741,"I just looked at the date we bought this, it w...",negative,0.9995
938,"Over the years, before today, we had purchased...",positive,0.9986
1159,"Like so many others, I've recently started bak...",positive,0.9987
...,...,...,...
56008,"I am a fussy toast person, I like it just so.\...",positive,0.9988
58835,I own the Black & Decker TO1303SBD toaster ove...,positive,0.9975
59673,I'd give it 5 stars but for a toaster this exp...,negative,0.999
60085,"I wouldn't say that I'm a toast connoisseur, b...",positive,0.9988


### Reviews RoBERTa Base

In [14]:
model_rvrb = "AnkitAI/reviews-roberta-base-sentiment-analysis"

sentiment_pipeline_rvrb = pipeline(
    "sentiment-analysis",
    model=model_rvrb,
    tokenizer=model_rvrb,
    device=device_id,
    truncation=True,        # Handles long reviews (truncates to 512 tokens)
    max_length=512,
    batch_size=32,          # Process in batches for speed
)
# Analyze sentiment and add results to the DataFrame
toaster_dedup_df = analyze_sentiment(
    toaster_dedup_df,
    sentiment_pipeline=sentiment_pipeline_rvrb,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

# Rename BERT output columns for clarity
toaster_dedup_df = toaster_dedup_df.rename(
    columns={
        "SENTIMENT": "RVRB_SENT",
        "SENTIMENT_SCORE": "RVRB_SCORE",
    }
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Analyzing 61395 texts (skipping 619 empty/invalid)...

Short texts (direct batch): 61305
Long texts (chunking + voting): 90



Long texts: 100%|██████████| 90/90 [00:04<00:00, 21.48it/s]


Done. 90 text(s) used chunking + voting.


In [15]:
# Check distribution of sentiment labels and review the long reviews that were chunked
print("\nSentiment Distribution:")
display(toaster_dedup_df["RVRB_SENT"].value_counts())

print("\nLong reviews that used chunking:")
display(toaster_dedup_df[toaster_dedup_df["CHUNKED"]][["RV_TRANS", "RVRB_SENT", "RVRB_SCORE"]])


Sentiment Distribution:


RVRB_SENT
positive    40038
negative    21357
Name: count, dtype: int64


Long reviews that used chunking:


,RV_TRANS,RVRB_SENT,RVRB_SCORE
3,BE AWARE: Read all the instructions included a...,negative,0.9912
196,[UPDATED at bottom of review]\n---------------...,negative,0.9838
741,"I just looked at the date we bought this, it w...",negative,0.9996
938,"Over the years, before today, we had purchased...",positive,0.9982
1159,"Like so many others, I've recently started bak...",positive,0.9973
...,...,...,...
56008,"I am a fussy toast person, I like it just so.\...",positive,0.9958
58835,I own the Black & Decker TO1303SBD toaster ove...,positive,0.9684
59673,I'd give it 5 stars but for a toaster this exp...,negative,0.9928
60085,"I wouldn't say that I'm a toast connoisseur, b...",positive,0.9748


## Results

In [16]:
# Review the results
cols = ["RV_TRANS", "SRVS", "CP_RVS", "TWRB_SENT", "TWRB_SCORE", "SRB_SENT", "SRB_SCORE", "RVRB_SENT", "RVRB_SCORE"]
pd.set_option("display.max_colwidth", 120)
display(toaster_dedup_df[cols].head())

,RV_TRANS,SRVS,CP_RVS,TWRB_SENT,TWRB_SCORE,SRB_SENT,SRB_SCORE,RVRB_SENT,RVRB_SCORE
0,Love making 4 slices at a time.,positive,0.6369,positive,0.967,positive,0.9988,positive,0.9982
1,Great item to use if you love grilled cheese but you have to use thin sliced bread only downfall,positive,0.8519,positive,0.8974,positive,0.9982,positive,0.9938
2,I had high hopes for this toaster - but it takes two cycles to brown properly.,positive,0.4404,negative,0.7371,negative,0.9995,negative,0.9986
3,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,positive,0.9914,neutral,0.6821,positive,0.9978,negative,0.9912
4,.,.,.,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Save Results to File

In [17]:
toaster_dedup_df.to_csv(os.path.join(data_processed_dir, "toaster_sentiment.csv"), index=False)

# Compare Results

In [20]:
# Extract relevant columns for comparison
compare_df = toaster_dedup_df[["SRVS", "TWRB_SENT", "SRB_SENT", "RV_TRANS", "RVRB_SENT"]].copy()

# Normalize text labels
compare_df["SRVS_clean"] = (
    compare_df["SRVS"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["TWRB_SENT_clean"] = (
    compare_df["TWRB_SENT"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["SRB_SENT_clean"] = (
    compare_df["SRB_SENT"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["RVRB_SENT_clean"] = (
    compare_df["RVRB_SENT"]
    .astype("string") 
    .str.strip()
    .str.lower()
)

In [22]:
# Keep only rows where both columns are available
compare_valid = compare_df.dropna(subset=["SRVS_clean", "TWRB_SENT_clean", "SRB_SENT_clean"]).copy()

# Check match between SRVS and TWRB_SENT
compare_valid["MATCH_SRVS_TWRB"] = (
    compare_valid["SRVS_clean"] == compare_valid["TWRB_SENT_clean"]
)

# Check match between SRVS and SRB_SENT
compare_valid["MATCH_SRVS_SRB"] = (
    compare_valid["SRVS_clean"] == compare_valid["SRB_SENT_clean"]
)

# Check match between TWRB_SENT and SRB_SENT
compare_valid["MATCH_TWRB_SRB"] = (
    compare_valid["TWRB_SENT_clean"] == compare_valid["SRB_SENT_clean"]
)

# Check match between SRVS and RVRB_SENT
compare_valid["MATCH_SRVS_RVRB"] = (
    compare_valid["SRVS_clean"] == compare_valid["RVRB_SENT_clean"]
)

# Check match between TWRB_SENT and RVRB_SENT 
compare_valid["MATCH_TWRB_RVRB"] = (
    compare_valid["TWRB_SENT_clean"] == compare_valid["RVRB_SENT_clean"]
)

# Check match betwween all three
compare_valid["MATCH_ALL"] = (
    (compare_valid["SRVS_clean"] == compare_valid["TWRB_SENT_clean"]) &
    (compare_valid["SRVS_clean"] == compare_valid["SRB_SENT_clean"]) &
    (compare_valid["SRVS_clean"] == compare_valid["RVRB_SENT_clean"])
)

print(f"Number of valid rows: {len(compare_valid)}")

Number of valid rows: 61395


## VADER vs Twitter RoBERTa Base

In [23]:
# Agreement rate between SRVS and TWRB_SENT
agreement_rate_SRVS_TWRB = compare_valid["MATCH_SRVS_TWRB"].mean()

print(f"Agreement rate between SRVS and TWRB_SENT: {agreement_rate_SRVS_TWRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["TWRB_SENT_clean"],
    margins=True
)

Agreement rate between SRVS and TWRB_SENT: 72.77%


TWRB_SENT_clean,negative,neutral,positive,All
SRVS_clean,,,,
negative,6561,835,762,8158
neutral,2541,2847,1377,6765
positive,8022,3183,35267,46472
All,17124,6865,37406,61395


## VADER vs RoBERTa Large English

In [24]:
# Agreement rate between SRVS and SRB_SENT
agreement_rate_SRVS_SRB = compare_valid["MATCH_SRVS_SRB"].mean()

print(f"Agreement rate between SRVS and SRB_SENT: {agreement_rate_SRVS_SRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["SRB_SENT_clean"],
    margins=True
)

Agreement rate between SRVS and SRB_SENT: 70.62%


SRB_SENT_clean,negative,positive,All
SRVS_clean,,,
negative,6932,1226,8158
neutral,3960,2805,6765
positive,10047,36425,46472
All,20939,40456,61395


## Twitter RoBERTa Base vs RoBERTa Large English

In [25]:
# Agreement rate between TWRB_SENT and SRB_SENT
agreement_rate_TWRB_SRB = compare_valid["MATCH_TWRB_SRB"].mean()

print(f"Agreement rate between TWRB_SENT and SRB_SENT: {agreement_rate_TWRB_SRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["TWRB_SENT_clean"],
    compare_valid["SRB_SENT_clean"],
    margins=True
)

Agreement rate between TWRB_SENT and SRB_SENT: 84.54%


SRB_SENT_clean,negative,positive,All
TWRB_SENT_clean,,,
negative,16039,1085,17124
neutral,3357,3508,6865
positive,1543,35863,37406
All,20939,40456,61395


## VADER vs Reviews RoBERTa Base

In [26]:
# Agreement rate between SRVS and RVRB_SENT
agreement_rate_SRVS_RVRB = compare_valid["MATCH_SRVS_RVRB"].mean()

print(f"Agreement rate between SRVS and RVRB_SENT: {agreement_rate_SRVS_RVRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["RVRB_SENT_clean"],
    margins=True
)

Agreement rate between SRVS and RVRB_SENT: 70.21%


RVRB_SENT_clean,negative,positive,All
SRVS_clean,,,
negative,6945,1213,8158
neutral,4100,2665,6765
positive,10312,36160,46472
All,21357,40038,61395


## Twitter RoBERTa Base vs Reviews ROBERTa Base

In [27]:
# Agreement rate between TWRB_SENT and RVRB_SENT
agreement_rate_TWRB_RVRB = compare_valid["MATCH_TWRB_RVRB"].mean()

print(f"Agreement rate between TWRB_SENT and RVRB_SENT: {agreement_rate_TWRB_RVRB:.2%}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["TWRB_SENT_clean"],
    compare_valid["RVRB_SENT_clean"],
    margins=True
)

Agreement rate between TWRB_SENT and RVRB_SENT: 84.31%


RVRB_SENT_clean,negative,positive,All
TWRB_SENT_clean,,,
negative,16065,1059,17124
neutral,3583,3282,6865
positive,1709,35697,37406
All,21357,40038,61395


## All Three

In [28]:
# Agreement rate where all three match
agreement_rate_all = compare_valid["MATCH_ALL"].mean()
print(f"Agreement rate where all three match: {agreement_rate_all:.2%}")

# Cross-tab comparison
pd.crosstab(
    [compare_valid["SRB_SENT_clean"],
    compare_valid["TWRB_SENT_clean"]],
     compare_valid["SRVS_clean"],
    margins=True
)

Agreement rate where all three match: 64.71%


SRVS_clean                      negative  neutral  positive    All
SRB_SENT_clean TWRB_SENT_clean                                    
negative       negative             6280     2477      7282  16039
               neutral               532     1399      1426   3357
               positive              120       84      1339   1543
positive       negative              281       64       740   1085
               neutral               303     1448      1757   3508
               positive              642     1293     33928  35863
All                                 8158     6765     46472  61395